# Notes

In [ ]:
%load_ext autoreload

%autoreload 2

: 

In [ ]:
cd ..

: 

In [4]:
from pathlib import Path 
from typing import Callable

import xarray as xr
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from geoarches.lightning_modules import load_module
from geoarches.dataloaders.era5 import Era5Forecast, Era5Dataset

In [5]:
ds = Era5Forecast(
    path="data/era5_240/full",  # default path
    domain="test", # domain to consider. domain = 'test' loads the 2020 period
    load_prev=True,  # whether to load previous state
    norm_scheme="pangu",  # default normalization scheme
    lead_time_hours=6
)

4it [00:00, 15.52it/s]

start time 2019-12-31T18:00:00


In [6]:
# this is 366 days in 6h steps, 
# because the model needs two input states,
# first doesn't have the prev
# and last doesn't have the next
# we only get 1462 available states
print(len(ds))

def get_init_state(date, step):
    pass

1462


In [7]:
from datetime import datetime, timedelta

def dates_2020_6h():
    start = datetime(2020, 1, 1, 0)
    end = datetime(2021, 1, 1, 0)

    dates = []
    current = start
    while current < end:
        dates.append(f"{current:%Y-%m-%d} - {current.hour}h")
        current += timedelta(hours=6)
        
    # removes first and last
    return dates[1:-1]

len(dates_2020_6h())

1462

In [8]:
path = "data/era5_240/full/era5_240_2020_12h.nc"

xr_ds = xr.open_dataset(path, engine="netcdf4")
xr_ds

<xarray.Dataset> Size: 5GB
Dimensions:                                           (time: 366,
                                                       longitude: 240,
                                                       latitude: 121, level: 13)
Coordinates:
  * time                                              (time) datetime64[ns] 3kB ...
  * longitude                                         (longitude) float64 2kB ...
  * latitude                                          (latitude) float64 968B ...
  * level                                             (level) int64 104B 50 ....
Data variables: (12/38)
    10m_u_component_of_wind                           (time, longitude, latitude) float32 43MB ...
    10m_v_component_of_wind                           (time, longitude, latitude) float32 43MB ...
    10m_wind_speed                                    (time, longitude, latitude) float32 43MB ...
    2m_temperature                                    (time, longitude, latitude) float32 43MB ...
    angle_of_sub_gridscale_orography                  (longitude, latitude) float32 116kB ...
    anisotropy_of_sub_gridscale_orography             (longitude, latitude) float32 116kB ...
    ...                                                ...
    type_of_high_vegetation                           (longitude, latitude) float32 116kB ...
    type_of_low_vegetation                            (longitude, latitude) float32 116kB ...
    u_component_of_wind                               (time, level, longitude, latitude) float32 553MB ...
    v_component_of_wind                               (time, level, longitude, latitude) float32 553MB ...
    vertical_velocity                                 (time, level, longitude, latitude) float32 553MB ...
    wind_speed                                        (time, level, longitude, latitude) float32 553MB ...

In [9]:
sample_state = ds[0]
gt_state = sample_state["next_state"]
gt_state

TensorDict(
    fields={
        level: Tensor(shape=torch.Size([6, 13, 121, 240]), device=cpu, dtype=torch.float32, is_shared=False),
        surface: Tensor(shape=torch.Size([4, 1, 121, 240]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)

In [18]:
ds[0].values()

dict_values([tensor(1577858400, dtype=torch.int32), TensorDict(
    fields={
        level: Tensor(shape=torch.Size([6, 13, 121, 240]), device=cpu, dtype=torch.float32, is_shared=False),
        surface: Tensor(shape=torch.Size([4, 1, 121, 240]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False), tensor(6, dtype=torch.int32), TensorDict(
    fields={
        level: Tensor(shape=torch.Size([6, 13, 121, 240]), device=cpu, dtype=torch.float32, is_shared=False),
        surface: Tensor(shape=torch.Size([4, 1, 121, 240]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False), TensorDict(
    fields={
        level: Tensor(shape=torch.Size([6, 13, 121, 240]), device=cpu, dtype=torch.float32, is_shared=False),
        surface: Tensor(shape=torch.Size([4, 1, 121, 240]), device=cpu, dtype=torch.float32, is_shared=False)},
    batch_size=torch.Size([]),
    de

In [34]:
start_ts_idx = 0
ds.timestamps[start_ts_idx][2]

np.datetime64('2020-01-01T00:00:00.000000000')

In [32]:
import torch
import numpy as np
from tensordict import TensorDict

timestamp_np = np.datetime64("2020-09-02T12:00:00.000000000")
timestamp_s = timestamp_np.astype("datetime64[s]").astype(np.int64)
timestamp = torch.tensor([timestamp_s])

state_with_time = sample_state["state"].unsqueeze(0)

ds.convert_to_xarray(state_with_time, timestamp)

<xarray.Dataset> Size: 10MB
Dimensions:                  (time: 1, level: 13, latitude: 121, longitude: 240)
Coordinates:
  * time                     (time) datetime64[s] 8B 2020-09-02T12:00:00
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * latitude                 (latitude) float64 968B 90.0 88.5 ... -88.5 -90.0
  * longitude                (longitude) float64 2kB 0.0 1.5 3.0 ... 357.0 358.5
Data variables:
    geopotential             (time, level, latitude, longitude) float32 2MB dask.array<chunksize=(1, 13, 121, 240), meta=np.ndarray>
    u_component_of_wind      (time, level, latitude, longitude) float32 2MB dask.array<chunksize=(1, 13, 121, 240), meta=np.ndarray>
    v_component_of_wind      (time, level, latitude, longitude) float32 2MB dask.array<chunksize=(1, 13, 121, 240), meta=np.ndarray>
    temperature              (time, level, latitude, longitude) float32 2MB dask.array<chunksize=(1, 13, 121, 240), meta=np.ndarray>
    specific_humidity        (time, level, latitude, longitude) float32 2MB dask.array<chunksize=(1, 13, 121, 240), meta=np.ndarray>
    vertical_velocity        (time, level, latitude, longitude) float32 2MB dask.array<chunksize=(1, 13, 121, 240), meta=np.ndarray>
    10m_u_component_of_wind  (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>
    10m_v_component_of_wind  (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>
    2m_temperature           (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>
    mean_sea_level_pressure  (time, latitude, longitude) float32 116kB dask.array<chunksize=(1, 121, 240), meta=np.ndarray>

In [8]:
# normalized space
gt_state["surface"][0, 0]

tensor([[0.0876, 0.0924, 0.0978,  ..., 0.0633, 0.0726, 0.0811],
        [0.4812, 0.4790, 0.4762,  ..., 0.4705, 0.4774, 0.4810],
        [0.8301, 0.8148, 0.7979,  ..., 0.8425, 0.8455, 0.8413],
        ...,
        [0.7402, 0.7859, 0.8261,  ..., 0.6194, 0.6524, 0.6942],
        [0.8845, 0.8783, 0.8716,  ..., 0.8933, 0.8901, 0.8871],
        [0.9881, 0.9848, 0.9798,  ..., 0.9880, 0.9885, 0.9888]])

## plotting stuff

In [ ]:
VARIABLES = {
    "surface": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_temperature",
        "mean_sea_level_pressure"
    ],
    "level": [
        "geopotential",
        "u_component_of_wind",
        "v_component_of_wind",
        "temperature",
        "specific_humidity",
        "vertical_velocity"
    ]
}

def get_last_experiment_id():
    paths = Path("results").glob("2026*")
    paths = sorted(paths)
    return paths[-1]

def read_experiment_tensor(id, t, partition):
    path = Path(id, str(t), f"{partition}.pt")
    tensor = torch.load(path)
    return tensor.cpu()

from mpl_toolkits.axes_grid1 import make_axes_locatable
def plot_sample(ground_truth, pred_state, tensor, title=""):
    ground_truth = ground_truth.cpu()
    pred_state = pred_state.cpu()
    tensor = tensor.cpu()

    combined = pred_state + tensor
    gen_error = combined - ground_truth
    gen_rmse = torch.sqrt(torch.mean(gen_error**2))
    print(f"Combined RMSE: {gen_rmse.item()}")

    det_error = pred_state - ground_truth
    det_rmse = torch.sqrt(torch.mean(det_error**2))
    print(f"Det RMSE: {det_rmse.item()}")

    fig, axes = plt.subplots(1, 6, dpi=1000, figsize=(20, 4))
    fig.suptitle(title)

    fields = [
        ("tensor", tensor),
        ("pred_state", pred_state),
        ("pred_state + tensor", combined),
        ("ground_truth", ground_truth),
        ("det_error", det_error),
        ("gen_error", gen_error),
    ]

    for ax, (name, field) in zip(axes, fields):
        vmax = field.abs().max().item()
        if vmax == 0:
            vmax = 1e-8
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

        im = ax.imshow(field, cmap="RdBu_r", norm=norm)
        ax.set_title(name)
        ax.set_xticks([])
        ax.set_yticks([])

        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        fig.colorbar(im, cax=cax)

    plt.tight_layout()
    plt.show()
    

def plot_state(ground_truth, pred_state, id, t, partition: str, var: str, var_idx: int, level: int):
    tensor = read_experiment_tensor(id, t, partition)
    tensor = tensor[var_idx, level]
    # print("tensor mean", tensor.mean())
    # print("pred_state mean", pred_state.mean())
    # print("ground_truth mean", ground_truth.mean())
    # print("tensor std", tensor.std())
    # print("pred_state std", pred_state.std())
    # print("ground_truth std", ground_truth.std())
    pred_state = pred_state[var_idx, level]

    title = f"{t} - {partition} - {var} - {level}"
    plot_sample(ground_truth, pred_state, tensor, title)

id = get_last_experiment_id()
partition = "surface"
var = "2m_temperature"
var_idx = VARIABLES[partition].index(var)
level = 0

ground_truth = gt_state[partition][var_idx, level]
print(ground_truth.mean())
pred_state = read_experiment_tensor(id, "pred_state", partition)
# print(pred_state.shape)

print(pred_state.mean())

for t in range(num_steps):
    plot_state(ground_truth, pred_state, id, t, partition, var, var_idx, level)

## scheduler

In [5]:
# steps
len(torch.tensor([1000.0000,  958.3750,  916.7500,  875.1250,  833.5000,  791.8750,
         750.2500,  708.6250,  667.0000,  625.3750,  583.7500,  542.1250,
         500.5000,  458.8750,  417.2500,  375.6250,  334.0000,  292.3750,
         250.7500,  209.1250,  167.5000,  125.8750,   84.2500,   42.6250,
           1.0000]))

25

In [7]:
T = 25
num_train_timesteps = 1000

timesteps = torch.linspace(
    num_train_timesteps, 1, T
)
timesteps

tensor([1000.0000,  958.3750,  916.7500,  875.1250,  833.5000,  791.8750,
         750.2500,  708.6250,  667.0000,  625.3750,  583.7500,  542.1250,
         500.5000,  458.8750,  417.2500,  375.6250,  334.0000,  292.3750,
         250.7500,  209.1250,  167.5000,  125.8750,   84.2500,   42.6250,
           1.0000])

In [21]:
from tqdm import tqdm
for i in tqdm(range(len(timesteps))):
    print(i)
    t = timesteps[i]

    s_t = t / num_train_timesteps   
    print(f"s_t = {s_t.item():.12f}")
    
    if i < len(timesteps) - 1:
        t_next = timesteps[i + 1]
        s_next = t_next / num_train_timesteps
        dt = s_t - s_next
    else:
        dt = s_t

    print(f"dt = {dt.item():.12f}")

100%|██████████| 25/25 [00:00<00:00, 15772.80it/s]

0
s_t = 1.000000000000
dt = 0.041625022888
1
s_t = 0.958374977112
dt = 0.041624963284
2
s_t = 0.916750013828
dt = 0.041625022888
3
s_t = 0.875124990940
dt = 0.041624963284
4
s_t = 0.833500027657
dt = 0.041625022888
5
s_t = 0.791875004768
dt = 0.041625022888
6
s_t = 0.750249981880
dt = 0.041624963284
7
s_t = 0.708625018597
dt = 0.041625022888
8
s_t = 0.666999995708
dt = 0.041625022888
9
s_t = 0.625374972820
dt = 0.041624963284
10
s_t = 0.583750009537
dt = 0.041625022888
11
s_t = 0.542124986649
dt = 0.041624963284
12
s_t = 0.500500023365
dt = 0.041625022888
13
s_t = 0.458875000477
dt = 0.041624993086
14
s_t = 0.417250007391
dt = 0.041624993086
15
s_t = 0.375625014305
dt = 0.041625022888
16
s_t = 0.333999991417
dt = 0.041624993086
17
s_t = 0.292374998331
dt = 0.041624993086
18
s_t = 0.250750005245
dt = 0.041625007987
19
s_t = 0.209124997258
dt = 0.041624993086
20
s_t = 0.167500004172
dt = 0.041625007987
21
s_t = 0.125874996185
dt = 0.041624993086
22
s_t = 0.084250003099
dt = 0.04162500426

In [ ]:

def tensordict_to_xarray(tensordict):
    ...


def save_state(array: xr.DataArray):
    def get_timestamp():
        date, time = str(datetime.now().replace(microsecond=0)).split(" ")
        timestamp = date + "_" + time
        return timestamp
    timestamp = get_timestamp()
    path = Path("results", f"{timestamp}")
    path.mkdir(parents=True, exist_ok=True)
    array.to_netcdf(path)

def load_state(path: Path):
    return xr.open_dataset(path, engine="netcdf4")

No.

Following formula get's us the masked term in dernormaliozed space:

x_denorm_masked = tensordict_apply(torch.mul, mask, x_denorm)

avg_over_mask = (
    sum(v.sum() for v in x_denorm_masked.values())
    / sum(v.sum() for v in mask.values())
)

g_0 = avg_over_mask

# this is pseudocode again since we would need some tensordict magic here too
Then we compute g_1 = anchor + drift * np.abs(anchor)

Now g_1 is my target in denormalized space. 
How do I get the equivalent transformation for x_norm?
It has to be some equation that enables us to reverse engineer this ...

In [ ]:
start_ts_idx

: 